In [1]:
import sys
!{sys.executable} -m pip install h5py scikit-learn xgboost imbalanced-learn pandas numpy joblib --quiet
print('Librerías instaladas')

Librerías instaladas


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from xgboost import XGBClassifier
import joblib
import h5py
import os

print(f'numpy:      {np.__version__}')
print(f'pandas:     {pd.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print('Todo importado correctamente')

numpy:      1.23.5
pandas:     2.3.3
matplotlib: 3.7.2
Todo importado correctamente


In [3]:
def generar_motor_monofasico(n_normal=2000, n_falla=100, random_state=42):
    np.random.seed(random_state)

    # --- Registros normales ---
    normal = pd.DataFrame({
        'voltaje_V':       np.random.normal(120, 2, n_normal),
        'corriente_A':     np.random.normal(8, 0.5, n_normal),
        'temperatura_C':   np.random.normal(65, 5, n_normal),
        'velocidad_rpm':   np.random.normal(1750, 30, n_normal),
        'factor_potencia': np.random.normal(0.92, 0.02, n_normal),
        'vibracion_g':     np.random.normal(0.5, 0.1, n_normal),
        'falla':           0
    })

    # --- Registros con falla ---
    falla = pd.DataFrame({
        'voltaje_V':       np.random.normal(108, 5, n_falla),   # caída de voltaje
        'corriente_A':     np.random.normal(12, 1.5, n_falla),  # sobrecorriente
        'temperatura_C':   np.random.normal(95, 8, n_falla),    # sobrecalentamiento
        'velocidad_rpm':   np.random.normal(1600, 80, n_falla), # pérdida de velocidad
        'factor_potencia': np.random.normal(0.75, 0.05, n_falla),# factor bajo
        'vibracion_g':     np.random.normal(1.8, 0.4, n_falla), # vibración alta
        'falla':           1
    })

    df = pd.concat([normal, falla], ignore_index=True).sample(frac=1, random_state=random_state)
    df['tipo_motor'] = 'AC_monofasico'
    return df

df_mono = generar_motor_monofasico()

print('Dataset AC Monofásico generado')
print(f'   Registros: {df_mono.shape[0]}')
print(f'   Fallas:    {df_mono["falla"].sum()} ({df_mono["falla"].mean()*100:.1f}%)')
df_mono.head()

Dataset AC Monofásico generado
   Registros: 2100
   Fallas:    100 (4.8%)


,voltaje_V,corriente_A,temperatura_C,velocidad_rpm,factor_potencia,vibracion_g,falla,tipo_motor
1034,119.033627,8.029492,68.578671,1798.160679,0.919413,0.412188,0,AC_monofasico
1176,116.614086,7.266607,58.492060,1785.612330,0.945152,0.526202,0,AC_monofasico
67,122.007066,7.932860,66.486221,1782.916606,0.938514,0.543375,0,AC_monofasico
1330,123.061502,8.456886,62.306196,1838.291450,0.948214,0.534936,0,AC_monofasico
650,123.697912,7.546122,63.756921,1757.717603,0.898226,0.478059,0,AC_monofasico


In [7]:
import urllib.request
url = 'https://zenodo.org/records/4314249'
print('Visita esta URL manualmente para ver los archivos disponibles:')
print(url)

Visita esta URL manualmente para ver los archivos disponibles:
https://zenodo.org/records/4314249


In [8]:
import zipfile

zip_path = 'open_DC_motor.zip'

if not os.path.exists('dc_motor'):
    print('⬇Descomprimiendo...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('dc_motor/')
        print('Archivos extraídos:')
        for f in z.namelist()[:10]:
            print(f'   {f}')
else:
    print('Ya descomprimido')

# Ver contenido
print('\n Archivos en dc_motor/:')
for root, dirs, files in os.walk('dc_motor'):
    for f in files:
        ruta = os.path.join(root, f)
        size = os.path.getsize(ruta) / 1024 / 1024
        print(f'   {ruta} ({size:.1f} MB)')

⬇Descomprimiendo...
Archivos extraídos:
   01-Start 3V/
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_04_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_08_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_12_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_17_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_21_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_25_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_30_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_34_Analogico.hdf5
   01-Start 3V/MOTOR-DC_2020_12_02_17_25_39_Analogico.hdf5

 Archivos en dc_motor/:
   dc_motor/open_DC_motor-processing data.xlsx (0.2 MB)
   dc_motor/01-Start 3V/MOTOR-DC_2020_12_02_17_25_04_Analogico.hdf5 (2.4 MB)
   dc_motor/01-Start 3V/MOTOR-DC_2020_12_02_17_25_08_Analogico.hdf5 (2.4 MB)
   dc_motor/01-Start 3V/MOTOR-DC_2020_12_02_17_25_12_Analogico.hdf5 (2.4 MB)
   dc_motor/01-Start 3V/MOTOR-DC_2020_12_02_17_25_17_Analogico.hdf5 (2.4 MB)
   dc_motor/01-Start 3V/

In [9]:
# Explorar un archivo para ver qué señales contiene
archivo_muestra = 'dc_motor/01-Start 3V/MOTOR-DC_2020_12_02_17_25_04_Analogico.hdf5'

with h5py.File(archivo_muestra, 'r') as f:
    print('Datasets disponibles:')
    def ver(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f'   {name} — shape: {obj.shape} — dtype: {obj.dtype}')
    f.visititems(ver)

Datasets disponibles:
   open_DC_motor — shape: (102413, 3) — dtype: float64


In [10]:
# Ver qué contienen las 3 columnas
with h5py.File(archivo_muestra, 'r') as f:
    datos = f['open_DC_motor'][:]
    print(f'Shape: {datos.shape}')
    print(f'\nPrimeras 5 filas:')
    print(datos[:5])
    print(f'\nEstadísticas:')
    print(f'  Canal 0 — min: {datos[:,0].min():.3f} max: {datos[:,0].max():.3f} mean: {datos[:,0].mean():.3f}')
    print(f'  Canal 1 — min: {datos[:,1].min():.3f} max: {datos[:,1].max():.3f} mean: {datos[:,1].mean():.3f}')
    print(f'  Canal 2 — min: {datos[:,2].min():.3f} max: {datos[:,2].max():.3f} mean: {datos[:,2].mean():.3f}')

Shape: (102413, 3)

Primeras 5 filas:
[[ 0.79000932  1.48203062  2.97924481]
 [-0.3760932   1.49794407  2.97545359]
 [-0.45163213  1.49794407  2.97545359]
 [ 0.26793909  1.50749214  2.96945082]
 [-0.00618565  1.53295366  2.96976676]]

Estadísticas:
  Canal 0 — min: -3.414 max: 3.286 mean: 0.013
  Canal 1 — min: -1.163 max: 3.067 mean: 1.625
  Canal 2 — min: 1.976 max: 5.212 mean: 2.967


In [11]:
def extraer_features_dc(ruta_archivo, etiqueta_falla=0):
    with h5py.File(ruta_archivo, 'r') as f:
        datos = f['open_DC_motor'][:]
    
    corriente = datos[:, 0]
    velocidad = datos[:, 1]
    voltaje   = datos[:, 2]
    
    return {
        # Corriente
        'corriente_mean':  corriente.mean(),
        'corriente_std':   corriente.std(),
        'corriente_max':   corriente.max(),
        'corriente_min':   corriente.min(),
        'corriente_rms':   np.sqrt(np.mean(corriente**2)),
        # Velocidad
        'velocidad_mean':  velocidad.mean(),
        'velocidad_std':   velocidad.std(),
        'velocidad_max':   velocidad.max(),
        # Voltaje
        'voltaje_mean':    voltaje.mean(),
        'voltaje_std':     voltaje.std(),
        'voltaje_min':     voltaje.min(),
        # Falla
        'falla':           etiqueta_falla,
        'tipo_motor':      'DC_brushed'
    }

# Probar con un archivo
feat = extraer_features_dc(archivo_muestra)
print('Features extraídas:')
for k, v in feat.items():
    print(f'   {k}: {v:.4f}' if isinstance(v, float) else f'   {k}: {v}')

Features extraídas:
   corriente_mean: 0.0130
   corriente_std: 0.8506
   corriente_max: 3.2856
   corriente_min: -3.4136
   corriente_rms: 0.8507
   velocidad_mean: 1.6254
   velocidad_std: 0.1979
   velocidad_max: 3.0670
   voltaje_mean: 2.9672
   voltaje_std: 0.0595
   voltaje_min: 1.9762
   falla: 0
   tipo_motor: DC_brushed


In [12]:
# Mapeo de carpetas a etiqueta de falla
# 01-Start 3V = operación normal a 3V
# 02-Change 5V = cambio de voltaje (transición normal)
# Carpetas con voltajes extremos o condiciones anómalas = falla

carpetas = []
for root, dirs, files in os.walk('dc_motor'):
    for f in files:
        if f.endswith('.hdf5'):
            carpetas.append(os.path.join(root, f))

print(f'Total archivos HDF5: {len(carpetas)}')

# Ver carpetas disponibles
dirs_disponibles = set([os.path.dirname(f).split('dc_motor/')[-1] for f in carpetas])
print('\nCarpetas:')
for d in sorted(dirs_disponibles):
    count = sum(1 for f in carpetas if d in f)
    print(f'   {d} — {count} archivos')

Total archivos HDF5: 377

Carpetas:
   01-Start 3V — 161 archivos
   02-Change 5V — 90 archivos
   03-Re_start 3V — 39 archivos
   04-Change 5V — 87 archivos


In [13]:
registros = []

etiquetas = {
    '01-Start 3V':    0,  # Normal
    '02-Change 5V':   1,  # Falla
    '03-Re_start 3V': 0,  # Normal
    '04-Change 5V':   1,  # Falla
}

print('⬇Procesando archivos HDF5...')
for ruta in carpetas:
    carpeta = os.path.dirname(ruta).split('dc_motor/')[-1]
    etiqueta = etiquetas.get(carpeta, 0)
    try:
        feat = extraer_features_dc(ruta, etiqueta)
        registros.append(feat)
    except Exception as e:
        print(f'   Error en {ruta}: {e}')

df_dc = pd.DataFrame(registros)

print(f'Dataset DC Brushed construido')
print(f'   Registros: {df_dc.shape[0]}')
print(f'   Fallas:    {df_dc["falla"].sum()} ({df_dc["falla"].mean()*100:.1f}%)')
print(f'   Normales:  {(df_dc["falla"]==0).sum()}')
df_dc.head()

⬇Procesando archivos HDF5...
Dataset DC Brushed construido
   Registros: 377
   Fallas:    177 (46.9%)
   Normales:  200


,corriente_mean,corriente_std,corriente_max,corriente_min,corriente_rms,velocidad_mean,velocidad_std,velocidad_max,voltaje_mean,voltaje_std,voltaje_min,falla,tipo_motor
0,0.013028,0.850609,3.285593,-3.413648,0.850709,1.625365,0.197938,3.067010,2.967154,0.059523,1.976151,0,DC_brushed
1,0.003729,0.879535,2.874119,-3.794311,0.879543,1.616711,0.195775,3.738558,2.966653,0.058799,1.040036,0,DC_brushed
2,-0.001843,0.916701,3.177067,-4.150109,0.916703,1.620359,0.197914,2.818760,2.965179,0.061692,1.514570,0,DC_brushed
3,-0.000907,0.912142,3.118124,-4.143220,0.912143,1.617788,0.200248,7.258613,2.964325,0.061985,1.318690,0,DC_brushed
4,-0.001967,0.954022,2.969634,-4.024225,0.954024,1.612117,0.205821,5.966441,2.964467,0.064241,-0.399364,0,DC_brushed


In [14]:
import urllib.request

# Descargar solo el archivo ZIP completo desde Figshare
url = 'https://figshare.com/ndownloader/articles/27216219/versions/1'
archivo = 'ac_trifasico.zip'

if not os.path.exists(archivo):
    print(' Descargando dataset AC Trifásico desde Figshare...')
    print('   Esto puede tardar varios minutos (5.16 GB)...')
    urllib.request.urlretrieve(url, archivo)
    print('Descarga completada')
else:
    print('Archivo ya existe')

 Descargando dataset AC Trifásico desde Figshare...
   Esto puede tardar varios minutos (5.16 GB)...
Descarga completada


In [15]:
import zipfile

if not os.path.exists('ac_trifasico'):
    print('Descomprimiendo...')
    with zipfile.ZipFile('ac_trifasico.zip', 'r') as z:
        z.extractall('ac_trifasico/')
        print('Archivos extraídos:')
        for f in z.namelist():
            print(f'   {f}')
else:
    print('Ya descomprimido')

Descomprimiendo...
Archivos extraídos:
   FILE 1.mat
   FILE 2.mat
   FILE 3.mat
   FILE 4.mat
   FILE 5.mat
   FILE 6.mat
   FILE 7.mat
   FILE 8.mat
   FILE 9.mat
   FILE 10.mat
   LABEL DATASET.mat
   FILE 1.csv
   FILE 2.csv
   FILE 3.csv
   FILE 4.csv
   FILE 5.csv
   FILE 6.csv
   FILE 7.csv
   FILE 8.csv
   FILE 9.csv
   FILE 10.csv
   LABEL DATASET.csv


In [16]:
# Leer archivo de etiquetas
df_labels = pd.read_csv('ac_trifasico/LABEL DATASET.csv')
print('=== LABEL DATASET ===')
print(df_labels.head(20))
print(f'\nColumnas: {df_labels.columns.tolist()}')
print(f'Shape: {df_labels.shape}')

=== LABEL DATASET ===
           0         1         2         3         4         5         6  \
0  -0.004840  0.000000  1.000000  0.022660  0.017167  0.018540  9.615406   
1  -0.000409  0.004834  1.000000  0.011673  0.015107  0.019227  9.401776   
2  -0.006317  0.000000  1.000610  0.007553  0.012360  0.019227  9.249184   
3  -0.001886  0.003223  1.000610  0.008927  0.017853  0.025407  9.706961   
4  -0.003363  0.000000  1.001221  0.007553  0.014420  0.013733  9.340739   
5   0.001068  0.001611  1.000610  0.009613  0.016480  0.013047  9.432295   
6  -0.006317  0.003223  1.000000  0.015107  0.008927  0.022660  9.584887   
7  -0.000409  0.003223  1.000610  0.010987  0.009613  0.012360  9.493332   
8  -0.006317  0.001611  1.000610  0.010300  0.006867  0.015793  9.706961   
9   0.001068  0.001611  1.000000  0.011673  0.014420  0.024033  9.371258   
10 -0.004840 -0.003223  1.000000  0.002747  0.003433  0.010987  9.462813   
11 -0.001886  0.001611  1.001221  0.013047  0.005493  0.019913  9.

In [17]:
# El dataset tiene 9000 columnas de señal + columna 'label'
# Cada fila = una ventana de señal completa
# Extraemos features estadísticas de cada fila

df_tri_raw = pd.read_csv('ac_trifasico/LABEL DATASET.csv')

print(f'Shape: {df_tri_raw.shape}')
print(f'Etiquetas únicas: {df_tri_raw["label"].unique()}')
print(f'Distribución:\n{df_tri_raw["label"].value_counts()}')

Shape: (19982, 9001)
Etiquetas únicas: [ 1  2  3  4  5  6  7  8  9 10 11 12 13]
Distribución:
label
3     2197
9     2037
5     1999
6     1999
7     1999
11    1999
12    1999
13    1999
1     1698
10    1059
4      799
2       99
8       99
Name: count, dtype: int64


In [18]:
# Según la documentación del dataset:
# Clase 1 = Normal (operación sin falla)
# Clases 2-13 = diferentes tipos de falla
# (desalineación, falla de fase, falla mecánica, etc.)

mapeo_falla = {
    1:  0,  # Normal
    2:  1,  # Falla
    3:  1,  # Falla
    4:  1,  # Falla
    5:  1,  # Falla
    6:  1,  # Falla
    7:  1,  # Falla
    8:  1,  # Falla
    9:  1,  # Falla
    10: 1,  # Falla
    11: 1,  # Falla
    12: 1,  # Falla
    13: 1,  # Falla
}

# Separar señales y etiquetas
y_raw = df_tri_raw['label'].map(mapeo_falla).values
X_raw = df_tri_raw.drop(columns=['label']).values

print(f'Total registros: {len(y_raw)}')
print(f'Normales: {(y_raw==0).sum()} ({(y_raw==0).mean()*100:.1f}%)')
print(f'Fallas:   {(y_raw==1).sum()} ({(y_raw==1).mean()*100:.1f}%)')

# Extraer features estadísticas de cada fila (ventana de señal)
print('\n Extrayendo features estadísticas...')

registros_tri = []
for i in range(len(X_raw)):
    señal = X_raw[i]
    registros_tri.append({
        'señal_mean':     señal.mean(),
        'señal_std':      señal.std(),
        'señal_max':      señal.max(),
        'señal_min':      señal.min(),
        'señal_rms':      np.sqrt(np.mean(señal**2)),
        'señal_kurtosis': float(pd.Series(señal).kurtosis()),
        'señal_skew':     float(pd.Series(señal).skew()),
        'señal_p25':      np.percentile(señal, 25),
        'señal_p75':      np.percentile(señal, 75),
        'señal_iqr':      np.percentile(señal, 75) - np.percentile(señal, 25),
        'falla':          y_raw[i],
        'tipo_motor':     'AC_trifasico'
    })

df_tri = pd.DataFrame(registros_tri)

print(f'Dataset AC Trifásico construido')
print(f'   Registros: {df_tri.shape[0]}')
print(f'   Features:  {df_tri.shape[1]-2}')
df_tri.head()

Total registros: 19982
Normales: 1698 (8.5%)
Fallas:   18284 (91.5%)

 Extrayendo features estadísticas...
Dataset AC Trifásico construido
   Registros: 19982
   Features:  10


,señal_mean,señal_std,señal_max,señal_min,señal_rms,señal_kurtosis,señal_skew,señal_p25,señal_p75,señal_iqr,falla,tipo_motor
0,-0.010316,4.649913,9.920591,-10.516739,4.649924,1.449723,-0.151706,-0.007794,0.021973,0.029768,0,AC_trifasico
1,-0.009732,4.650272,9.920591,-10.455702,4.650282,1.449164,-0.148797,-0.007794,0.021973,0.029768,0,AC_trifasico
2,-0.009670,4.651039,10.012146,-10.455702,4.651049,1.450376,-0.151494,-0.006454,0.021973,0.028428,0,AC_trifasico
3,-0.012481,4.653986,10.012146,-10.455702,4.654003,1.450889,-0.155313,-0.006317,0.023347,0.029664,0,AC_trifasico
4,-0.012186,4.658115,10.012146,-10.455702,4.658131,1.451004,-0.154808,-0.007794,0.022145,0.029939,0,AC_trifasico


In [19]:
# Estandarizar columnas comunes para unificar
# Usaremos solo las features que podemos calcular para los 3 tipos

df_mono_std = pd.DataFrame({
    'mean':      df_mono['voltaje_V'],
    'std':       df_mono['corriente_A'],
    'max':       df_mono['temperatura_C'],
    'min':       df_mono['velocidad_rpm'],
    'rms':       df_mono['vibracion_g'],
    'kurtosis':  df_mono['factor_potencia'],
    'skew':      df_mono['corriente_A'] / df_mono['voltaje_V'],
    'p25':       df_mono['temperatura_C'] / df_mono['velocidad_rpm'],
    'p75':       df_mono['vibracion_g'] * df_mono['factor_potencia'],
    'iqr':       df_mono['corriente_A'] - df_mono['vibracion_g'],
    'falla':     df_mono['falla'],
    'tipo_motor': 'AC_monofasico'
})

df_dc_std = pd.DataFrame({
    'mean':      df_dc['corriente_mean'],
    'std':       df_dc['corriente_std'],
    'max':       df_dc['corriente_max'],
    'min':       df_dc['corriente_min'],
    'rms':       df_dc['corriente_rms'],
    'kurtosis':  df_dc['velocidad_mean'],
    'skew':      df_dc['velocidad_std'],
    'p25':       df_dc['voltaje_mean'],
    'p75':       df_dc['voltaje_std'],
    'iqr':       df_dc['voltaje_min'],
    'falla':     df_dc['falla'],
    'tipo_motor': 'DC_brushed'
})

df_tri_std = pd.DataFrame({
    'mean':      df_tri['señal_mean'],
    'std':       df_tri['señal_std'],
    'max':       df_tri['señal_max'],
    'min':       df_tri['señal_min'],
    'rms':       df_tri['señal_rms'],
    'kurtosis':  df_tri['señal_kurtosis'],
    'skew':      df_tri['señal_skew'],
    'p25':       df_tri['señal_p25'],
    'p75':       df_tri['señal_p75'],
    'iqr':       df_tri['señal_iqr'],
    'falla':     df_tri['falla'],
    'tipo_motor': 'AC_trifasico'
})

# Unificar
df_priora = pd.concat([df_mono_std, df_dc_std, df_tri_std], ignore_index=True)

print(' Dataset Priora unificado')
print(f'   Total registros: {df_priora.shape[0]}')
print(f'   Features:        {df_priora.shape[1]-2}')
print()
print('--- Por tipo de motor ---')
for tipo in df_priora['tipo_motor'].unique():
    sub = df_priora[df_priora['tipo_motor']==tipo]
    print(f'{tipo}:')
    print(f'   Total:   {len(sub)}')
    print(f'   Normales: {(sub["falla"]==0).sum()}')
    print(f'   Fallas:   {(sub["falla"]==1).sum()}')

df_priora.head()

 Dataset Priora unificado
   Total registros: 22459
   Features:        10

--- Por tipo de motor ---
AC_monofasico:
   Total:   2100
   Normales: 2000
   Fallas:   100
DC_brushed:
   Total:   377
   Normales: 200
   Fallas:   177
AC_trifasico:
   Total:   19982
   Normales: 1698
   Fallas:   18284


,mean,std,max,min,rms,kurtosis,skew,p25,p75,iqr,falla,tipo_motor
0,119.033627,8.029492,68.578671,1798.160679,0.412188,0.919413,0.067456,0.038138,0.378971,7.617304,0,AC_monofasico
1,116.614086,7.266607,58.492060,1785.612330,0.526202,0.945152,0.062313,0.032757,0.497341,6.740406,0,AC_monofasico
2,122.007066,7.932860,66.486221,1782.916606,0.543375,0.938514,0.065020,0.037291,0.509965,7.389485,0,AC_monofasico
3,123.061502,8.456886,62.306196,1838.291450,0.534936,0.948214,0.068721,0.033894,0.507234,7.921949,0,AC_monofasico
4,123.697912,7.546122,63.756921,1757.717603,0.478059,0.898226,0.061004,0.036273,0.429405,7.068063,0,AC_monofasico


In [20]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Codificar tipo de motor
le_tipo = LabelEncoder()
df_priora['tipo_motor_cod'] = le_tipo.fit_transform(df_priora['tipo_motor'])

print('Codificación de tipos:')
for i, clase in enumerate(le_tipo.classes_):
    print(f'   {i} = {clase}')

# Features y variables objetivo
features = ['mean', 'std', 'max', 'min', 'rms', 
            'kurtosis', 'skew', 'p25', 'p75', 'iqr', 'tipo_motor_cod']

X = df_priora[features].values
y_falla = df_priora['falla'].values
y_tipo  = df_priora['tipo_motor_cod'].values

# División 80/20
X_train, X_test, y_train, y_test, y_tipo_train, y_tipo_test = train_test_split(
    X, y_falla, y_tipo, test_size=0.2, random_state=42, stratify=y_falla
)

# Normalización
scaler_priora = StandardScaler()
X_train_sc = scaler_priora.fit_transform(X_train)
X_test_sc  = scaler_priora.transform(X_test)

print(f'\n Preprocesamiento completado')
print(f'   Entrenamiento: {X_train.shape[0]} muestras')
print(f'   Prueba:        {X_test.shape[0]} muestras')
print(f'   Features:      {X_train.shape[1]}')

Codificación de tipos:
   0 = AC_monofasico
   1 = AC_trifasico
   2 = DC_brushed

 Preprocesamiento completado
   Entrenamiento: 17967 muestras
   Prueba:        4492 muestras
   Features:      11


In [21]:
# MODELO 1: Identificador de tipo de motor (multiclase)
print(' Entrenando identificador de tipo de motor...')

rf_tipo = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf_tipo.fit(X_train_sc, y_tipo_train)
y_pred_tipo = rf_tipo.predict(X_test_sc)

print('\n=== IDENTIFICACIÓN DE TIPO DE MOTOR ===')
print(f'Accuracy: {accuracy_score(y_tipo_test, y_pred_tipo)*100:.2f}%')
print()
print(classification_report(y_tipo_test, y_pred_tipo,
      target_names=le_tipo.classes_))

 Entrenando identificador de tipo de motor...

=== IDENTIFICACIÓN DE TIPO DE MOTOR ===
Accuracy: 100.00%

               precision    recall  f1-score   support

AC_monofasico       1.00      1.00      1.00       405
 AC_trifasico       1.00      1.00      1.00      4010
   DC_brushed       1.00      1.00      1.00        77

     accuracy                           1.00      4492
    macro avg       1.00      1.00      1.00      4492
 weighted avg       1.00      1.00      1.00      4492



In [22]:
# MODELO 2: Detector de fallas (binario)
from imblearn.over_sampling import SMOTE

print('Balanceando clases con SMOTE...')
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)
print(f'   Normales: {(y_train_sm==0).sum()}')
print(f'   Fallas:   {(y_train_sm==1).sum()}')

print('\n Entrenando detector de fallas...')
xgb_falla = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_falla.fit(X_train_sm, y_train_sm)
y_pred_falla = xgb_falla.predict(X_test_sc)

print('\n=== DETECCIÓN DE FALLAS ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_falla)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_falla)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_falla,
      target_names=['Normal', 'Falla']))

Balanceando clases con SMOTE...
   Normales: 14849
   Fallas:   14849

 Entrenando detector de fallas...

=== DETECCIÓN DE FALLAS ===
Accuracy: 99.98%
F1-Score: 99.99%

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       780
       Falla       1.00      1.00      1.00      3712

    accuracy                           1.00      4492
   macro avg       1.00      1.00      1.00      4492
weighted avg       1.00      1.00      1.00      4492



In [23]:
def priora_diagnostico(datos_sensor, modelo_tipo, modelo_falla, scaler, le_tipo):
    """
    Sistema Priora — Diagnóstico completo de motor eléctrico
    Entrada: diccionario con valores de sensores
    Salida: tipo de motor, estado y prioridad de mantenimiento
    """
    # Preparar datos
    X = np.array([[
        datos_sensor['mean'],
        datos_sensor['std'],
        datos_sensor['max'],
        datos_sensor['min'],
        datos_sensor['rms'],
        datos_sensor['kurtosis'],
        datos_sensor['skew'],
        datos_sensor['p25'],
        datos_sensor['p75'],
        datos_sensor['iqr'],
        datos_sensor['tipo_motor_cod']
    ]])

    X_sc = scaler.transform(X)

    # Identificar tipo de motor
    tipo_cod  = modelo_tipo.predict(X_sc)[0]
    tipo_nombre = le_tipo.inverse_transform([tipo_cod])[0]

    # Detectar falla
    pred_falla = modelo_falla.predict(X_sc)[0]
    prob_falla = modelo_falla.predict_proba(X_sc)[0][1]

    # Sistema de priorización
    if prob_falla < 0.20:
        prioridad = '🟢 NORMAL'
        accion    = 'Motor operando correctamente'
        nivel     = 1
    elif prob_falla < 0.50:
        prioridad = '🟡 ATENCIÓN'
        accion    = 'Incrementar frecuencia de monitoreo'
        nivel     = 2
    elif prob_falla < 0.80:
        prioridad = '🟠 RIESGO'
        accion    = 'Programar mantenimiento a corto plazo'
        nivel     = 3
    else:
        prioridad = '🔴 CRÍTICO'
        accion    = 'Detener motor para inspección inmediata'
        nivel     = 4

    return {
        'tipo_motor':   tipo_nombre,
        'falla':        bool(pred_falla),
        'probabilidad': round(prob_falla * 100, 2),
        'prioridad':    prioridad,
        'nivel':        nivel,
        'accion':       accion
    }

print('Sistema Priora cargado correctamente')

Sistema Priora cargado correctamente


In [24]:
# Prueba con un motor de cada tipo

motores_prueba = [
    {
        'nombre': 'Motor AC Monofásico — Normal',
        'datos': {
            'mean': 119.5, 'std': 8.1, 'max': 67.0, 'min': 1780.0,
            'rms': 0.45, 'kurtosis': 0.92, 'skew': 0.065,
            'p25': 0.037, 'p75': 0.42, 'iqr': 7.5,
            'tipo_motor_cod': 0
        }
    },
    {
        'nombre': 'Motor DC Brushed — Falla',
        'datos': {
            'mean': 0.8, 'std': 1.9, 'max': 5.2, 'min': -4.1,
            'rms': 1.9, 'kurtosis': 3.8, 'skew': 1.2,
            'p25': 3.8, 'p75': 0.15, 'iqr': -1.2,
            'tipo_motor_cod': 2
        }
    },
    {
        'nombre': 'Motor AC Trifásico — Falla',
        'datos': {
            'mean': -0.5, 'std': 6.2, 'max': 12.5, 'min': -13.0,
            'rms': 6.2, 'kurtosis': 2.8, 'skew': -0.3,
            'p25': -0.05, 'p75': 0.04, 'iqr': 0.09,
            'tipo_motor_cod': 1
        }
    },
]

print('='*60)
print('        SISTEMA PRIORA — DIAGNÓSTICO DE MOTORES')
print('='*60)

for motor in motores_prueba:
    resultado = priora_diagnostico(
        motor['datos'], rf_tipo, xgb_falla,
        scaler_priora, le_tipo
    )
    print(f'\n {motor["nombre"]}')
    print(f'   Tipo identificado:  {resultado["tipo_motor"]}')
    print(f'   Probabilidad falla: {resultado["probabilidad"]}%')
    print(f'   Prioridad:          {resultado["prioridad"]}')
    print(f'   Acción:             {resultado["accion"]}')
    print('-'*60)

        SISTEMA PRIORA — DIAGNÓSTICO DE MOTORES

 Motor AC Monofásico — Normal
   Tipo identificado:  AC_monofasico
   Probabilidad falla: 0.0%
   Prioridad:          🟢 NORMAL
   Acción:             Motor operando correctamente
------------------------------------------------------------

 Motor DC Brushed — Falla
   Tipo identificado:  DC_brushed
   Probabilidad falla: 92.97%
   Prioridad:          🔴 CRÍTICO
   Acción:             Detener motor para inspección inmediata
------------------------------------------------------------

 Motor AC Trifásico — Falla
   Tipo identificado:  AC_trifasico
   Probabilidad falla: 99.97%
   Prioridad:          🔴 CRÍTICO
   Acción:             Detener motor para inspección inmediata
------------------------------------------------------------


In [25]:
# Guardar todos los modelos y objetos necesarios
joblib.dump(rf_tipo,       'priora_modelo_tipo.pkl')
joblib.dump(xgb_falla,     'priora_modelo_falla.pkl')
joblib.dump(scaler_priora, 'priora_scaler.pkl')
joblib.dump(le_tipo,       'priora_label_encoder.pkl')

print(' Modelos guardados:')
print('   priora_modelo_tipo.pkl')
print('   priora_modelo_falla.pkl')
print('   priora_scaler.pkl')
print('   priora_label_encoder.pkl')

 Modelos guardados:
   priora_modelo_tipo.pkl
   priora_modelo_falla.pkl
   priora_scaler.pkl
   priora_label_encoder.pkl


In [26]:
print('='*60)
print('     VALIDACIÓN FINAL — SISTEMA PRIORA')
print('='*60)

# 1. Verificar datasets
print('\n DATASETS')
print(f'   AC Monofásico:  {len(df_mono_std):,} registros | Fallas: {df_mono_std["falla"].sum()} ({df_mono_std["falla"].mean()*100:.1f}%)')
print(f'   DC Brushed:     {len(df_dc_std):,} registros | Fallas: {df_dc_std["falla"].sum()} ({df_dc_std["falla"].mean()*100:.1f}%)')
print(f'   AC Trifásico:   {len(df_tri_std):,} registros | Fallas: {df_tri_std["falla"].sum()} ({df_tri_std["falla"].mean()*100:.1f}%)')
print(f'   Total Priora:   {len(df_priora):,} registros')

# 2. Verificar modelos
print('\n MODELOS')
y_pred_tipo_val  = rf_tipo.predict(X_test_sc)
y_pred_falla_val = xgb_falla.predict(X_test_sc)

print(f'   Identificador de tipo:  Accuracy {accuracy_score(y_tipo_test, y_pred_tipo_val)*100:.2f}%')
print(f'   Detector de fallas:     Accuracy {accuracy_score(y_test, y_pred_falla_val)*100:.2f}% | F1 {f1_score(y_test, y_pred_falla_val)*100:.2f}%')

# 3. Verificar sistema de priorización
print('\n SISTEMA DE PRIORIZACIÓN')
probs = xgb_falla.predict_proba(X_test_sc)[:, 1]
print(f'   🟢 NORMAL   (0-20%):   {(probs < 0.20).sum():,} motores')
print(f'   🟡 ATENCIÓN (20-50%):  {((probs >= 0.20) & (probs < 0.50)).sum():,} motores')
print(f'   🟠 RIESGO   (50-80%):  {((probs >= 0.50) & (probs < 0.80)).sum():,} motores')
print(f'   🔴 CRÍTICO  (80-100%): {(probs >= 0.80).sum():,} motores')

# 4. Verificar función priora
print('\n FUNCIÓN PRIORA')
test_normal = {
    'mean': 119.5, 'std': 8.1, 'max': 67.0, 'min': 1780.0,
    'rms': 0.45, 'kurtosis': 0.92, 'skew': 0.065,
    'p25': 0.037, 'p75': 0.42, 'iqr': 7.5, 'tipo_motor_cod': 0
}
r = priora_diagnostico(test_normal, rf_tipo, xgb_falla, scaler_priora, le_tipo)
print(f'   Test motor normal: {r["prioridad"]} — {r["tipo_motor"]} ' if r['nivel']==1 else f'   ⚠️ Error en test normal')

print('\n' + '='*60)
print(' TODO EN ORDEN — Sistema listo para guardar')
print('='*60)

     VALIDACIÓN FINAL — SISTEMA PRIORA

 DATASETS
   AC Monofásico:  2,100 registros | Fallas: 100 (4.8%)
   DC Brushed:     377 registros | Fallas: 177 (46.9%)
   AC Trifásico:   19,982 registros | Fallas: 18284 (91.5%)
   Total Priora:   22,459 registros

 MODELOS
   Identificador de tipo:  Accuracy 100.00%
   Detector de fallas:     Accuracy 99.98% | F1 99.99%

 SISTEMA DE PRIORIZACIÓN
   🟢 NORMAL   (0-20%):   779 motores
   🟡 ATENCIÓN (20-50%):  0 motores
   🟠 RIESGO   (50-80%):  0 motores
   🔴 CRÍTICO  (80-100%): 3,713 motores

 FUNCIÓN PRIORA
   Test motor normal: 🟢 NORMAL — AC_monofasico 

 TODO EN ORDEN — Sistema listo para guardar


In [27]:
joblib.dump(rf_tipo,       'priora_modelo_tipo.pkl')
joblib.dump(xgb_falla,     'priora_modelo_falla.pkl')
joblib.dump(scaler_priora, 'priora_scaler.pkl')
joblib.dump(le_tipo,       'priora_label_encoder.pkl')

print(' Modelos guardados:')
print('   priora_modelo_tipo.pkl')
print('   priora_modelo_falla.pkl')
print('   priora_scaler.pkl')
print('   priora_label_encoder.pkl')

 Modelos guardados:
   priora_modelo_tipo.pkl
   priora_modelo_falla.pkl
   priora_scaler.pkl
   priora_label_encoder.pkl
